# Tensorflow2.0 小练习

In [2]:
import tensorflow as tf
import numpy as np

## 实现softmax函数

In [6]:
def softmax(x):
    ##########
    '''实现softmax函数，只要求对最后一维归一化，
    不允许用tf自带的softmax函数'''
    
    # 1. 为了数值稳定性，减去最后一维的最大值 (防止 exp 溢出)
    # keepdims=True 保证形状不变，以便进行广播减法
    x_max = np.max(x, axis=-1, keepdims=True)
    x_shifted = x - x_max
    
    # 2. 计算指数
    exp_x = np.exp(x_shifted)
    
    # 3. 计算最后一维的和
    sum_exp_x = np.sum(exp_x, axis=-1, keepdims=True)
    
    # 4. 归一化得到概率分布
    prob_x = exp_x / sum_exp_x
    ##########
    return prob_x

test_data = np.random.normal(size=[10, 5])
(softmax(test_data) - tf.nn.softmax(test_data, axis=-1).numpy())**2 <0.0001

array([[ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True]])

## 实现sigmoid函数

In [8]:
def sigmoid(x):
    ##########
    '''实现sigmoid函数， 不允许用tf自带的sigmoid函数'''
    ##########
    pos_mask = (x >= 0)
    neg_mask = (x < 0)
    
    prob_x = np.zeros_like(x, dtype=np.float64)
    
    # 处理非负部分
    z = np.exp(-x[pos_mask])
    prob_x[pos_mask] = 1 / (1 + z)
    
    # 处理负数部分
    z = np.exp(x[neg_mask])
    prob_x[neg_mask] = z / (1 + z)
    return prob_x

test_data = np.random.normal(size=[10, 5])
(sigmoid(test_data) - tf.nn.sigmoid(test_data).numpy())**2 < 0.0001

array([[ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True]])

## 实现 softmax 交叉熵loss函数

In [9]:
def softmax_ce(x, label):
    ##########
    '''实现 softmax 交叉熵loss函数， 不允许用tf自带的softmax_cross_entropy函数'''
    epsilon = 1e-7
    x_clipped = tf.clip_by_value(x, epsilon, 1.0 - epsilon)
    

    ce_per_sample = -tf.reduce_sum(label * tf.math.log(x_clipped), axis=1)
    

    loss = tf.reduce_mean(ce_per_sample)
    ##########
    return loss

test_data = np.random.normal(size=[10, 5])
prob = tf.nn.softmax(test_data)
label = np.zeros_like(test_data)
label[np.arange(10), np.random.randint(0, 5, size=10)]=1.

((tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(label, test_data))
  - softmax_ce(prob, label))**2 < 0.0001).numpy()

True

## 实现 sigmoid 交叉熵loss函数

In [13]:
def sigmoid_ce(x, label):
    ##########
    '''实现 softmax 交叉熵loss函数， 不允许用tf自带的softmax_cross_entropy函数'''
    epsilon = 1e-7
    x_clipped = tf.clip_by_value(x, epsilon, 1.0 - epsilon)
    
    label = tf.cast(label, x_clipped.dtype)
    
    loss_per_sample = -(label * tf.math.log(x_clipped) + 
                        (1 - label) * tf.math.log(1 - x_clipped))
    
    # 3. 返回平均损失
    loss = tf.reduce_mean(loss_per_sample)
    ##########
    return loss

test_data = np.random.normal(size=[10])
prob = tf.nn.sigmoid(test_data)
label = np.random.randint(0, 2, 10).astype(test_data.dtype)
print (label)

((tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(label, test_data))- sigmoid_ce(prob, label))**2 < 0.0001).numpy()


[1. 0. 0. 1. 1. 1. 0. 0. 0. 1.]


True